# Lab 4 · Inside the model: probabilities, perplexity and attention

**Time:** 30 min · **You will:** take `generate()` apart — read the model's raw next-token probabilities, see a hand-written generation loop, measure how "surprised" the model is by a text (perplexity), and look at what its attention connects.

*How to use this notebook: **Kernel → Restart Kernel and Run All Cells**, read each result, then change the `🔧` values and run those cells again (Shift+Enter).*

In [ ]:
import sys, os
# find the course folder (the one holding slm_lab.py), whichever folder this notebook was opened from
sys.path.insert(0, next(os.path.abspath(d) for d in ("..", "../..") if os.path.exists(os.path.join(d, "slm_lab.py"))))
from slm_lab import *
import torch.nn.functional as F

# "eager" attention lets us read the attention weights in 4.4 (the fast kernels don't return them)
tok, model = load(attn_implementation="eager")

## 4.1 One forward pass = one probability distribution
Give the model a text and it returns **logits**: one score per vocabulary token, at every position. The scores at the **last** position, passed through softmax, are the probabilities of the next token.

In [ ]:
text = "The capital of India is"
enc = tok(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    logits = model(**enc).logits          # shape: (batch, sequence length, vocabulary size)
print("logits shape:", tuple(logits.shape))

### 🔧 Try it 4.1 — Top-k next tokens
`top_next(text, k)` lists the `k` most likely next tokens with their probabilities.
*Steps in the code:* tokenize → forward pass → take `logits[0, -1]` → `softmax` → `torch.topk` → decode each ID.

🔧 **Try:** change `my_text` to *"ABC Company employees get paternity leave of"*. The model happily ranks numbers it has no basis for — the root of hallucination.

In [ ]:
@torch.no_grad()
def top_next(text, k=5):
    enc = tok(text, return_tensors="pt").to(model.device)
    probs = F.softmax(model(**enc).logits[0, -1].float(), dim=-1)
    top = torch.topk(probs, k)
    return [(tok.decode(int(i)), float(p)) for p, i in zip(top.values, top.indices)]

my_text = "The capital of India is"   # 🔧 any start of a sentence
how_many = 5                          # 🔧 how many guesses to show

for t, p in top_next(my_text, how_many):
    print(f"{p:6.1%}  {t!r}")

In [ ]:
res = top_next("The capital of India is", 5)
with torch.no_grad():
    _ids = tok("The capital of India is", return_tensors="pt").to(model.device)
    _best = tok.decode(int(model(**_ids).logits[0, -1].argmax()))
check(len(res) == 5 and res[0][0] == _best
      and all(res[i][1] >= res[i + 1][1] for i in range(4)) and 0 < sum(p for _, p in res) <= 1.0001,
      f"Top guess {res[0][0]!r} ({res[0][1]:.0%}). Generation is just: pick from this list, append, repeat.",
      "top_next did not return the most likely tokens in order.")

### 🔧 Try it 4.2 — generate() by hand
`my_generate` is greedy generation written out: repeat `n` times → forward pass, take the **argmax** (most likely token) of the last logits, append that ID to the input. The cell compares it with the library's `model.generate`.

🔧 **Try:** change `start` to your own sentence start, or `n_tokens` to `20`.

In [ ]:
@torch.no_grad()
def my_generate(text, n=8):
    ids = tok(text, return_tensors="pt").input_ids.to(model.device)
    for _ in range(n):
        next_id = model(ids).logits[0, -1].argmax()          # the most likely next token
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)    # append it and go again
    return tok.decode(ids[0, -n:])

start = "The three primary colours are"   # 🔧 any start of a sentence
n_tokens = 8                              # 🔧 how many tokens to add

mine = my_generate(start, n_tokens)
ref_ids = model.generate(**tok(start, return_tensors="pt").to(model.device),
                         max_new_tokens=n_tokens, do_sample=False)
reference = tok.decode(ref_ids[0, -n_tokens:])
print("mine     :", repr(mine))
print("generate :", repr(reference))

In [ ]:
check(mine == reference, "Identical. model.generate is this loop plus conveniences (stopping, sampling, KV cache).",
      "the two differ - with long outputs, tiny rounding differences on a GPU can change a token.")

Notice that the loop re-reads the **whole** sequence every step. `generate` uses a **KV cache** to avoid that — Lab 16 measures the difference.

## 4.3 Perplexity: how surprised is the model?
If you pass `labels=input_ids`, the model returns the average **cross-entropy loss** of predicting each token from the ones before it. **Perplexity = e^loss**: roughly "how many tokens the model was choosing between". Lower = the text is more predictable to the model. This is exactly the quantity training minimises (Extra Lab C).

### 🔧 Try it 4.3 — Compare perplexities
🔧 **Try:** add your own line to `texts`, e.g. `"my text": "…",` — a sentence in Hindi, a line of code, or random letters.

In [ ]:
@torch.no_grad()
def perplexity(text):
    enc = tok(text, return_tensors="pt").to(model.device)
    loss = model(**enc, labels=enc["input_ids"]).loss
    return float(torch.exp(loss))

texts = {                                                                # 🔧 change or add texts
    "normal": "Employees are entitled to paid leave every year.",
    "scrambled": "year leave every paid Employees to entitled are.",
    "abc fact": "ABC Company employees get 20 days of paternity leave.",
}
ppl = {name: perplexity(t) for name, t in texts.items()}
for name, v in ppl.items():
    print(f"{name:10} perplexity {v:8.1f}")

In [ ]:
check(ppl["scrambled"] > 3 * ppl["normal"],
      "Scrambled text is far more surprising. After fine-tuning (Extra Lab C) we'll check that ABC facts become "
      "much less surprising to the model.",
      "keep the 'normal' and 'scrambled' lines in texts to compare them.")

In [ ]:
save_report("lab3_perplexity", {"abc_fact_before_finetune": round(ppl["abc fact"], 2)})

## 4.4 Attention: which words does a word look at?
Each layer has several **attention heads**. For every token, a head produces weights over all previous tokens (each row sums to 1). Decoder models like ours only look **backwards** (causal attention).

🔧 **Try:** change `sentence` (keep the word `it` in it).

In [ ]:
sentence = "The animal didn't cross the street because it was too tired"   # 🔧 keep " it" in the sentence
enc = tok(sentence, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model(**enc, output_attentions=True)
words = [tok.decode(i) for i in enc["input_ids"][0]]
print(f"{len(out.attentions)} layers, each of shape {tuple(out.attentions[0].shape)}  (batch, heads, query, key)")
print(list(enumerate(words)))

### 👀 See it 4.4 — What does "it" attend to?
For the token `" it"`, the cell averages the attention **over all heads of every layer** and lists the 3 *earlier* tokens that get the most attention (skipping the very first token and `" it"` itself).

*Why skip the first token?* Models dump a lot of attention on the first token as a "no-op" (an **attention sink**), which hides the interesting pattern.

In [ ]:
if " it" in words:
    it_pos = words.index(" it")
    att = torch.stack(out.attentions)[:, 0].float().mean(dim=(0, 1))   # layers & heads averaged → (query, key)
    row = att[it_pos, 1:it_pos]                                        # earlier tokens, skipping the sink
    top = row.topk(min(3, len(row))).indices + 1
    top_words = [words[i] for i in top]
    print("'it' looks most at:", top_words)
else:
    top_words = []
    print("⚠️ The sentence has no ' it' - put the word 'it' back in the sentence above and run both cells.")

In [ ]:
check(len(top_words) == 3,
      "Look at what came out: averaged attention often favours nearby or high-signal words (here possibly "
      "'animal'/'street', 'because'). Individual heads specialise; the average blurs them.",
      "the sentence needs the word 'it' with at least 3 words before it.")

**Optional:** plot one head with `import matplotlib.pyplot as plt; plt.imshow(out.attentions[-1][0, 0].float().cpu())` and try other layers/heads.

## 🧠 Check your understanding
1. What exactly does the model output at each step, and how does `generate` turn that into text?
2. What does a perplexity of 2 vs 200 tell you about a text?
3. Why is generating with the `my_generate` loop slower than `model.generate`?

*✍️ Your answer:* 

**Next:** Lab 5 — measure how often the plain model is wrong (short, or homework). Then Lab 6.